In [26]:
import pandas as pd

Esse notebook utiliza o datatset aria midi unique como ponto de partida do projeto. O arquivo de metadata é analisado para recolher informações sobre os midis disponiveis para uso. Após tratativas, o datatset é separado em dados de treino e validação e posteriormente os arquivos estão separados entre essas duas categorias para facilitar a manipulação na etapa de tokenização.

In [46]:
pip install scikit-learn

In [29]:
with open("/content/MIDI-LSTM-and-transformer-decoder/metadata.json", "r") as f:
    df = pd.read_json(f)

In [30]:
df_metadata=pd.DataFrame()

for col_name, col_data in df.items():
    raw_values = pd.DataFrame([col_data.values[0]])
    raw_values["scores"] = [col_data.values[1]]
    col_name = str(col_name)
    name = col_name.zfill(6) if len(col_name) <6 else col_name
    raw_values["audio_name"] = name
    df_metadata = pd.concat([raw_values, df_metadata], ignore_index=True)
print(df_metadata)

          composer   opus      genre performer  music_period  \
0            liszt  392.0  classical   hamelin      romantic   
1           ligeti    NaN  classical       NaN  contemporary   
2      saint-saens   56.0  classical       NaN      romantic   
3       moszkowski   68.0  classical       NaN      romantic   
4         respighi   44.0  classical       NaN     classical   
...            ...    ...        ...       ...           ...   
32517     schubert  498.0  classical       NaN      romantic   
32518       franck   10.0  classical       NaN      romantic   
32519      purcell    5.0  classical       NaN       baroque   
32520     maykapar   15.0  classical       NaN           NaN   
32521        weber   77.0  classical       NaN     classical   

                          scores audio_name  piece_number     form  \
0      {'0': 0.9963000000000001}     207047           NaN      NaN   
1      {'0': 0.9904000000000001}     207046           5.0    etude   
2      {'0': 0.992300

In [32]:
df_metadata.columns

Index(['composer', 'opus', 'genre', 'performer', 'music_period', 'scores',
       'audio_name', 'piece_number', 'form', 'key_signature', 'difficulty'],
      dtype='object')

In [35]:
df_metadata.drop(columns=['opus','performer','key_signature','piece_number','form','difficulty'], inplace=True)

In [36]:
df_metadata.columns

Index(['composer', 'genre', 'music_period', 'scores', 'audio_name'], dtype='object')

In [37]:
cleaned_df_metadata = df_metadata.dropna(subset=['genre', 'music_period'])

print(cleaned_df_metadata)

          composer      genre  music_period                     scores  \
0            liszt  classical      romantic  {'0': 0.9963000000000001}   
1           ligeti  classical  contemporary  {'0': 0.9904000000000001}   
2      saint-saens  classical      romantic  {'0': 0.9923000000000001}   
3       moszkowski  classical      romantic  {'0': 0.9902000000000001}   
4         respighi  classical     classical  {'0': 0.9981000000000001}   
...            ...        ...           ...                        ...   
32516      arensky  classical      romantic  {'0': 0.9618000000000001}   
32517     schubert  classical      romantic               {'0': 0.991}   
32518       franck  classical      romantic              {'0': 0.9846}   
32519      purcell  classical       baroque              {'0': 0.9957}   
32521        weber  classical     classical  {'0': 0.9510000000000001}   

      audio_name  
0         207047  
1         207046  
2         207042  
3         207040  
4         207039

In [38]:
print(cleaned_df_metadata.groupby(["genre", "music_period"])["genre"].count())


genre       music_period 
atonal      contemporary        8
            modern             17
blues       contemporary        1
            modern              3
classical   baroque          2291
            classical        7818
            contemporary     1990
            impressionist     854
            modern           1034
            romantic         9817
folk        classical           1
            contemporary        4
            modern              5
            romantic            2
jazz        baroque             1
            classical           2
            contemporary        5
            modern             60
            romantic            1
ragtime     classical           2
            contemporary        4
            modern              2
rock        modern              1
soundtrack  contemporary        8
Name: genre, dtype: int64


In [43]:
print(cleaned_df_metadata.groupby(["genre", "composer"])["genre"].count())


genre       composer  
atonal      bacevicius    1
            berg          3
            ligeti        2
            nancarrow     1
            noland        1
                         ..
ragtime     zerkovitz     1
rock        norton        1
soundtrack  bemani        1
            o'halloran    5
            ohalloran     2
Name: genre, Length: 1888, dtype: int64


In [44]:
most_famous_genre = cleaned_df_metadata[cleaned_df_metadata['genre'] == "classical"].copy()

print(most_famous_genre)

          composer      genre  music_period                     scores  \
0            liszt  classical      romantic  {'0': 0.9963000000000001}   
1           ligeti  classical  contemporary  {'0': 0.9904000000000001}   
2      saint-saens  classical      romantic  {'0': 0.9923000000000001}   
3       moszkowski  classical      romantic  {'0': 0.9902000000000001}   
4         respighi  classical     classical  {'0': 0.9981000000000001}   
...            ...        ...           ...                        ...   
32516      arensky  classical      romantic  {'0': 0.9618000000000001}   
32517     schubert  classical      romantic               {'0': 0.991}   
32518       franck  classical      romantic              {'0': 0.9846}   
32519      purcell  classical       baroque              {'0': 0.9957}   
32521        weber  classical     classical  {'0': 0.9510000000000001}   

      audio_name  
0         207047  
1         207046  
2         207042  
3         207040  
4         207039

In [57]:
from sklearn.model_selection import train_test_split

X_train, X_test = train_test_split(most_famous_genre, test_size=0.33, random_state=42)

In [58]:
X_train


,composer,genre,music_period,scores,audio_name
9394,barjansky,classical,romantic,{'0': 0.9726},146697
14486,satie,classical,impressionist,{'0': 0.9367000000000001},114359
2308,strauss,classical,romantic,{'0': 0.994},192097
7495,cimarosa,classical,classical,{'0': 0.98},159514
18045,liszt,classical,romantic,{'0': 0.9858},091618
...,...,...,...,...,...
29449,schubert,classical,romantic,{'0': 0.9774},019120
7317,macdowell,classical,romantic,{'0': 0.9571000000000001},160630
1183,bartok,classical,classical,{'0': 0.9904000000000001},199374
21514,liszt,classical,romantic,{'0': 0.9939},070270


In [59]:
X_test

,composer,genre,music_period,scores,audio_name
17493,mozart,classical,classical,{'0': 0.9911000000000001},095174
31339,mozart,classical,classical,{'0': 0.9994000000000001},007366
3691,vladigerov,classical,classical,{'0': 0.9886},183335
20678,lemoine,classical,classical,{'0': 0.9994000000000001},075470
2652,prudent,classical,romantic,{'0': 0.9906},189873
...,...,...,...,...,...
8810,debussy,classical,impressionist,{'0': 0.9598000000000001},150697
26830,prokofiev,classical,modern,{'0': 0.9927},036070
22580,hannikainen,classical,romantic,{'0': 0.9923000000000001},063386
10290,mozart,classical,classical,{'0': 0.9538000000000001},140823


In [62]:
midi_filenames_train = (X_train["audio_name"].values).astype(str)
len(midi_filenames_train)

15948

In [63]:
midi_filenames_test = (X_test["audio_name"].values).astype(str)
len(midi_filenames_test)

7856

In [66]:
from pathlib import Path
import shutil
import zipfile

def copy_files(origin, destination, filenames):
  with zipfile.ZipFile(destination, "w", zipfile.ZIP_DEFLATED) as zipf:
    for path_f in origin.rglob('*'):
      if path_f.is_file() and path_f.name.split("_")[0] in filenames:
        zipf.write(path_f, path_f.relative_to(origin))

copy_files(Path("/content/data2"),Path("/content/MIDI-LSTM-and-transformer-decoder/dataset_training.zip"), midi_filenames_train)
copy_files(Path("/content/data2"),Path("/content/MIDI-LSTM-and-transformer-decoder/dataset_test.zip"), midi_filenames_test)



